# 05 — Auditoria de gênero, raça e interseções

Compara grupos anotados usando o mesmo detector, teste e limiar. Classe positiva = fake. **FPR**: real classificada como fake; **FNR**: fake aceita como real. Acurácia global pode esconder esses dois tipos de erro. A ausência de diferença detectável não prova equidade.

As categorias UTKFace são imperfeitas e não devem ser chamadas de identidade verdadeira. Tonalidade de pele não é sinônimo de raça.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', ROOT)


In [ ]:
%pip -q install scikit-learn pandas matplotlib scipy


In [ ]:
import numpy as np,pandas as pd
from sklearn.metrics import roc_auc_score
from math import sqrt

def wilson(k,n):
    if not n:return (np.nan,np.nan)
    z=1.95996398454;p=k/n;d=1+z*z/n
    center=(p+z*z/(2*n))/d;half=z*sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return center-half,center+half

def summarize(d):
    y=d.label.to_numpy();p=d.pred.to_numpy()
    tn=int(((y==0)&(p==0)).sum());fp=int(((y==0)&(p==1)).sum())
    fn=int(((y==1)&(p==0)).sum());tp=int(((y==1)&(p==1)).sum())
    div=lambda a,b:a/b if b else np.nan
    tpr=div(tp,tp+fn);tnr=div(tn,tn+fp)
    return dict(n=len(d),n_real=tn+fp,n_fake=tp+fn,tn=tn,fp=fp,fn=fn,tp=tp,
        accuracy=div(tp+tn,len(d)),balanced_accuracy=(tpr+tnr)/2,
        precision=div(tp,tp+fp),recall=tpr,f1=div(2*tp,2*tp+fp+fn),
        fpr=div(fp,fp+tn),fnr=div(fn,fn+tp),
        auc=roc_auc_score(y,d.p_fake) if len(set(y))==2 and 'p_fake' in d else np.nan)

def table_metrics(df):
    rows=[]
    for condition,data in df.groupby('condition'):
        for axis in ['all','gender','race','intersection']:
            groups=[('Todos',data)] if axis=='all' else data.groupby(axis,dropna=False)
            for group,d in groups:
                row=summarize(d);row.update(condition=condition,axis=axis,group=group)
                eligible=d[d.clean_pred==d.label] if 'clean_pred' in d else d.iloc[:0]
                row.update(asr=(eligible.pred!=eligible.label).mean() if len(eligible) else np.nan,
                           n_asr=len(eligible))
                rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
RESULT=ROOT/'resultados'/'xception_piloto_10'
path=RESULT/'predicoes_robustez.csv'
if not path.exists():path=RESULT/'predicoes_limpas.csv'
df=pd.read_csv(path)
assert not df.duplicated(['image_id','condition']).any()
for c in ['gender','race']:df[c]=df[c].fillna('unknown').astype(str)
df['intersection']=df.race+' | '+df.gender
clean=df[df.condition=='clean'][['image_id','pred']].rename(columns={'pred':'clean_pred'})
df=df.merge(clean,on='image_id',how='left',validate='many_to_one')
assert df.clean_pred.notna().all()
expected=set(clean.image_id)
assert all(set(d.image_id)==expected for _,d in df.groupby('condition')), 'Condições com amostras diferentes.'
summary=table_metrics(df)
summary.to_csv(RESULT/'metricas_por_grupo.csv',index=False)
display(summary)


## 1. Diferenças entre grupos
Exportamos a amplitude máximo–mínimo de cada métrica. Para equalized odds, usamos o maior valor entre a amplitude de FPR e a de TPR (= amplitude de FNR). Calculamos apenas quando todos os grupos comparados têm denominador válido. Grupo unknown aparece nas tabelas, mas é excluído das comparações de atributos conhecidos. Nenhum corte de tamanho transforma uma análise em prova de justiça.


In [ ]:
gaps=[]
for (condition,axis),t in summary[summary.axis!='all'].groupby(['condition','axis']):
    t=t[~t.group.str.contains('unknown',case=False)]
    row={'condition':condition,'axis':axis,'groups':len(t)}
    for metric in ['accuracy','fpr','fnr','asr']:
        row[metric+'_gap']=t[metric].max()-t[metric].min() if len(t)>=2 and t[metric].notna().all() else np.nan
    row['equalized_odds_gap']=max(row['fpr_gap'],row['fnr_gap']) if pd.notna(row['fpr_gap']) and pd.notna(row['fnr_gap']) else np.nan
    gaps.append(row)
pd.DataFrame(gaps).to_csv(RESULT/'disparidades.csv',index=False)
display(pd.DataFrame(gaps))


## 2. Incerteza por reamostragem de componentes
Real e fake do mesmo alvo são dependentes, assim como imagens que reutilizam uma fonte. Construímos componentes conectados entre unidades de fonte e alvo e reamostramos componentes inteiros. Sem IDs de pessoa, dependência entre fotos distintas da mesma pessoa continua desconhecida. Com menos de 10 componentes por grupo, não publicamos intervalo; a amostra precisa ser ampliada ou a geração redesenhada. Os intervalos são exploratórios, sem correção para comparações múltiplas.


In [ ]:
parent={}
def find(x):
    parent.setdefault(x,x)
    if parent[x]!=x:parent[x]=find(parent[x])
    return parent[x]
def union(a,b):parent[find(a)]=find(b)
for r in df[df.condition=='clean'].itertuples():union(r.unit_id,r.source_unit_id)
df['cluster']=df.unit_id.map(find)
rng=np.random.default_rng(42);B=500;cis=[]
for (condition,group),d in df.groupby(['condition','intersection']):
    clusters=list(d.cluster.unique());n=len(clusters)
    vals=[]
    if n>=10:
        chunks={c:d[d.cluster==c] for c in clusters}
        for _ in range(B):
            sample=pd.concat([chunks[c] for c in rng.choice(clusters,n,replace=True)])
            result=summarize(sample)
            eligible=sample[sample.clean_pred==sample.label]
            result['asr']=(eligible.pred!=eligible.label).mean() if len(eligible) else np.nan
            vals.append(result)
    for metric in ['accuracy','fpr','fnr','asr']:
        v=np.array([r[metric] for r in vals]);v=v[np.isfinite(v)]
        lo,hi=np.quantile(v,[.025,.975]) if len(v)>=.9*B else (np.nan,np.nan)
        cis.append(dict(condition=condition,group=group,metric=metric,clusters=n,low=lo,high=hi))
pd.DataFrame(cis).to_csv(RESULT/'intervalos_cluster.csv',index=False)
display(pd.DataFrame(cis))


## 3. Figuras para o pôster


In [ ]:
import matplotlib.pyplot as plt
for axis in ['gender','race','intersection']:
    t=summary[(summary.axis==axis)&(summary.condition=='clean')].set_index('group')
    fig,ax=plt.subplots(figsize=(max(8,len(t)*.8),5))
    t[['accuracy','fpr','fnr']].plot.bar(ax=ax,color=['#125b99','#16a59a','#dd8142'])
    ax.set_ylim(0,1);ax.set_ylabel('Proporção');ax.set_title('Teste limpo: '+axis)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=35,ha='right')
    plt.tight_layout();plt.savefig(RESULT/f'limpo_{axis}.png',dpi=220);plt.show()
fg=summary[(summary.axis=='intersection')&summary.condition.str.startswith('fgsm_')].copy()
if len(fg):
    fg['epsilon']=fg.condition.str.split('_').str[-1].astype(int)
    fig,ax=plt.subplots(figsize=(10,5))
    for g,d in fg.groupby('group'):
        d=d.sort_values('epsilon');ax.plot(d.epsilon,d.asr,marker='o',label=g)
    ax.set(xlabel='Epsilon (níveis /255)',ylabel='ASR',ylim=(0,1),title='FGSM por grupo anotado')
    ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
    plt.tight_layout();plt.savefig(RESULT/'fgsm_interseccional.png',dpi=220);plt.show()


## 4. Conclusão baseada na execução
Preencha o texto com `metricas_por_grupo.csv`, `disparidades.csv` e os intervalos. Informe N total, reais/fakes por grupo, fonte dos rótulos, split por arquivo/pessoa e técnica de manipulação. Descreva separadamente FPR e FNR. Não conclua que um modelo é “sem preconceito” a partir de acurácias parecidas. Para testar mitigação, treine outra condição sem modificar o teste e compare também desempenho global, incerteza e validação externa. O pipeline atual faz auditoria; não comprova mitigação.
